In [44]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import dash
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import dash_bootstrap_components as dbc
import base64
JupyterDash.infer_jupyter_proxy_config()


# Configure the plotting routines
import pandas as pd


# Import AnimalShelter
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
username = 'aacuser'
password = 'aacpass'

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
# Deprecated
app = JupyterDash(__name__,external_stylesheets=[dbc.themes.BOOTSTRAP])

image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    # Top Header
    dbc.Row([
        dbc.Col(html.A(children=html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()), style={'width': '120px'}), href='https://www.snhu.edu')),
        dbc.Col(html.Center(html.B(html.H1('Animal Rescue Dashboard')))),
        dbc.Col(dbc.Col([
            html.P('Author: Alan Abraham Puthenparambil Kochumon', style={'text-align': 'end', 'margin-right': '2rem'}),
            html.P('Unique Identifier: 80a1727895a438dea937994e6b6671da8b00bcb2', style={'text-align': 'end', 'margin-right': '2rem'})
        ])), # UID
    ], style={'margin-top': '24px', 'align-items': 'center'}),
    html.Hr(),
    html.Div(
    # Filters
    dbc.Col(
    [
        dbc.Label('Outcomes', html_for='center-outcomes-row', width=2, className='fw-bold'),
        dbc.Col(
            dbc.RadioItems(
                id='filter-type',
                options=[
                    {'label': 'Water Rescue', 'value': 'water'},
                    {'label': 'Mountain Rescue', 'value': 'mountain'},
                    {'label': 'Disaster Rescue', 'value': 'disaster'},
                    {'label': 'Reset', 'value': 'reset'},
                ],
                inline=True,
                className='mb-3',
                value='reset' # Default value
            ),
            width=10,
        ),
    ],
    className='ms-4 mb-3',
    )
    ),
    html.Hr(),
    # Data
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {'name': i, 'id': i, 'deletable': False, 'selectable': True} for i in df.columns
        ],
        data=df.to_dict('records'),
        editable=False,
        # Selection
        row_selectable='single',
        selected_rows=[0],
        # Filtering
        filter_action='native',
        # Sorting
        sort_action='native',
        sort_mode='multi',
        # Pagination
        page_action='native',
        page_current=0,
        page_size=15 
    ),
    html.Br(),
    html.Hr(),
    # Graph
    #This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6'),
        html.Div(
            id='map-id',
            className='col s12 m6')
        ])
])


#############################################
# Interaction Between Components / Controller
#############################################

    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    # Create a query based on the filter type
    if filter_type == 'water':
        query = {
            'animal_type': 'Dog',
            'breed': {'$in': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']},
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
    elif filter_type == 'mountain':
        query = {
            'animal_type': 'Dog',
            'breed': {'$in': ['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog', 'Siberian Husky', 'Rottweiler']},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
    elif filter_type == 'disaster':
        query = {
            'animal_type': 'Dog',
            'breed': {'$in': ['German Shepherd', 'Doberman Pinscher', 'Golden Retriever', 'Bloodhound', 'Rottweiler']},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}
        }
    else:
        query = {}
    
    # Get the results from database
    filtered_records = pd.DataFrame.from_records(db.read(query))
    
    # Drop the id column created by mongodb
    filtered_records.drop(columns=['_id'],inplace=True)
    
    return filtered_records.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')])
def update_graphs(viewData):
    # Return a hint text if there no data to show
    if viewData is None or len(viewData) == 0:
        return [html.P('No data available for chart rendering.')]
    
    dff = pd.DataFrame.from_records(viewData)
    fig = px.pie(dff, names='outcome_type', title='Outcome after Service')
    fig.update_layout(margin=dict(l=20, r=20, t=40, b=20))
    return [
        dcc.Graph(            
            figure = fig,
            style={'width': '100%', 'height': '100%'}
        )    
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    # Apply no style if there are no selected columns
    if selected_columns is None:
        return []
    # Else update the background color
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
    Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, index):
    dff = pd.DataFrame.from_dict(viewData)
    # Render message if dataframe is empty or None
    if dff is None or len(dff) == 0:
        return [html.P('No data found!')]
    
    # Because we only allow single row selection, the list can 
    # be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Check if the animal has no name and update its name accordingly
    animal_name = dff.iloc[row]['name']
    if len(animal_name) < 1:
        animal_name = 'Unnamed'
    
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
           center=[30.75,-97.48], zoom=10, children=[
           dl.TileLayer(id='base-layer-id'),
           # Marker with tool tip and popup
           # Column 13 and 14 define the grid-coordinates for 
           # the map
           # Column 4 defines the breed for the animal
           # Column 9 defines the name of the animal
           dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]],
              children=[
              dl.Tooltip(dff.iloc[row,4]),
              dl.Popup([
                 html.H1('Animal Name'),
                html.P(animal_name)
             ])
          ])
       ])
    ]

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server()

Dash is running on http://127.0.0.1:8050/proxy/8050/

Dash app running on https://raymondnull-cabinetegypt-3000.codio.io/proxy/8050/
